In [ ]:
#!/usr/bin/env python3
import os
import torch
import numpy as np
from PIL import Image
from tqdm import tqdm

from cnet_dataset import HSIControlNetDataset
from diffusers import DDIMScheduler, UNet2DConditionModel
from transformers import AutoTokenizer, CLIPTextModel
from warp_and_controlnet_model import WarpAndControlNetModel

# Custom UNet with camera input
from model_with_camera import NewUNet2DConditionModel
import cv2

# ---------------- Config ----------------
ROOT_DIR     = "/data/train/"
CNET_CKPT    = "/data/nd_model_4_wrap_cnet/checkpoint-36000/controlnet"
UNET_WEIGHTS = "/home/viplab/liukejia/project/depth_model/nd_ckpt_diffuser/nd_mv_ema_diffuser.pth"
BASE_MODEL   = "stable-diffusion-v1-5/stable-diffusion-v1-5"
COMPARE_DIR  = "./test"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16  # inference dtype

# Must match training
NUM_FRAMES   = 4
LATENT_SIZE  = 32
HSI_CHANNELS = 48
CAMERA_DIM   = 16

os.makedirs(COMPARE_DIR, exist_ok=True)

# ---------------- 1. Load components ----------------
print("Loading models...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(BASE_MODEL, subfolder="text_encoder", torch_dtype=DTYPE).to(DEVICE)

controlnet = WarpAndControlNetModel.from_pretrained(CNET_CKPT, torch_dtype=DTYPE).to(DEVICE)
controlnet.warper = controlnet.warper.half()

temp_model = UNet2DConditionModel.from_pretrained(BASE_MODEL, subfolder="unet", low_cpu_mem_usage=True)
unet = NewUNet2DConditionModel(**temp_model.config, camera_dim=CAMERA_DIM)
unet_state = torch.load(UNET_WEIGHTS, map_location="cpu")
unet.load_state_dict(unet_state, strict=True)
unet.to(DEVICE, dtype=DTYPE)
unet.eval()
# controlnet = WarpAndControlNetModel.from_unet(unet,conditioning_channels=48).to(DEVICE, dtype=DTYPE)
controlnet.eval()
text_encoder.eval()

scheduler = DDIMScheduler.from_pretrained(BASE_MODEL, subfolder="scheduler")
scheduler.set_timesteps(20)

# ---------------- 2. Dataset ----------------
dataset = HSIControlNetDataset(root_dir=ROOT_DIR, image_size=512)

# ---------------- 3. Inference ----------------
@torch.no_grad()
def infer(sample, seed=42):
    # ===== 调试打印：当前样本对应的 4 个文件夹 =====
    folders = dataset.groups[dataset._index]          # 关键一行
    print(f"[debug] idx={dataset._index}  folders={folders}")
    # ===================================================
    generator = torch.Generator(DEVICE).manual_seed(seed)

    # Prompt embeddings (repeat per frame)
    prompt = sample["text"]
    text_input = tokenizer(
        prompt,
        padding="max_length",
        max_length=tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt",
    )
    prompt_embeds = text_encoder(text_input.input_ids.to(DEVICE))[0]
    prompt_embeds = prompt_embeds.repeat_interleave(NUM_FRAMES, dim=0)  # (4, 77, 768)

    # Conditioning image: expected (4, 48, 256, 256) from dataset
    cond_image = torch.from_numpy(sample["conditioning_image"]).to(DEVICE, dtype=DTYPE)
    assert cond_image.shape[0] == NUM_FRAMES and cond_image.shape[1] == HSI_CHANNELS, \
        f"Unexpected cond_image shape {cond_image.shape}"

    # Camera matrix: (4, 16) from dataset
    camera_matrix = torch.from_numpy(sample["camera_matrix"]).to(DEVICE, dtype=DTYPE)
    print("camera_matrix.shape",camera_matrix.shape)
    assert camera_matrix.shape == (NUM_FRAMES, CAMERA_DIM), \
        f"Unexpected camera_matrix shape {camera_matrix.shape}"

    # Latents init: noise, shape (4, 4, 32, 32)
    latents = torch.randn(
        (NUM_FRAMES, 4, LATENT_SIZE, LATENT_SIZE),
        generator=generator,
        device=DEVICE,
        dtype=DTYPE,
    )
    latents = latents * scheduler.init_noise_sigma

    guidance_scale = 5
    print(f"Running inference: {prompt}")
    for t in tqdm(scheduler.timesteps):
        
        if guidance_scale > 1.0:
            # Duplicate for classifier-free guidance
            latent_model_input = torch.cat([latents] * 2)
            prompt_embeds_input = torch.cat([torch.zeros_like(prompt_embeds), prompt_embeds])
            cond_image_input = torch.cat([cond_image, cond_image]) # TODO: check if cond image is zero
            camera_matrix_input = torch.cat([camera_matrix, camera_matrix])
        else:
            latent_model_input = latents
            prompt_embeds_input = prompt_embeds
            cond_image_input = cond_image
            camera_matrix_input = camera_matrix

        # ControlNet
        down_block_res_samples, mid_block_res_sample = controlnet(
            sample=latent_model_input,
            timestep=t,
            encoder_hidden_states=prompt_embeds_input,
            controlnet_cond=cond_image_input,
            return_dict=False,
        )

        # UNet with camera
        noise_pred = unet(
            sample=latent_model_input,
            timestep=t,
            encoder_hidden_states=prompt_embeds_input,
            down_block_additional_residuals=down_block_res_samples,
            mid_block_additional_residual=mid_block_res_sample,
            camera=camera_matrix_input,
        ).sample

        if guidance_scale > 1.0:
            noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
            noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

        latents = scheduler.step(noise_pred, t, latents).prev_sample

    # # Post-process: [-1,1] -> [0,1], upsample to 256 for viewing
    # image = (latents / 2 + 0.5).clamp(0, 1)
    # image = torch.nn.functional.interpolate(image, size=(256, 256), mode="bilinear")
    # image = image.cpu().permute(0, 2, 3, 1).float().numpy()  # (4, 256, 256, 4)

    # # Visualize first 3 channels per frame and concat horizontally
    # final_imgs = []
    # for i in range(NUM_FRAMES):
    #     img_np = (image[i, :, :, :3] * 255).astype(np.uint8)
    #     final_imgs.append(Image.fromarray(img_np))

    # w, h = final_imgs[0].size
    # grid = Image.new("RGB", (w * NUM_FRAMES, h))
    # for i, img in enumerate(final_imgs):
    #     grid.paste(img, (w * i, 0))
    # return grid
    print("maxmin",latents.max(),latents.min())
         # ---------- 后处理：[-1,1] -> [0,1]，上采样 256 ----------
    
    # image = (latents / 2 + 0.5).clamp(0, 1)
    image = latents.clamp(0,1)
    image = torch.nn.functional.interpolate(image, size=(256, 256), mode="bilinear")
    image = image.cpu().permute(0, 2, 3, 1).numpy()  # (4, 256, 256, 4)
    # 原始图像
    if "image" in sample:
        raw = np.array(sample["image"])              # (4,4,32,32)
        raw = raw[:, :4, ...].transpose(0, 2, 3, 1)  # (4,32,32,4)
    else:
        raw = None

    raw_row = []
    if raw is not None:
        for i in range(min(NUM_FRAMES, raw.shape[0])):
            frame = cv2.resize(raw[i], (256, 256), interpolation=cv2.INTER_LINEAR)  # 不要写回 raw[i]
            raw_np = (frame[..., :3] * 255).astype(np.uint8)
            raw_row.append(Image.fromarray(raw_np))

    w, h = raw_row[0].size if raw_row else (256, 256)
    raw_canvas = Image.new("RGB", (w * NUM_FRAMES, h))
    for i, im in enumerate(raw_row):
        raw_canvas.paste(im, (w * i, 0))
    # 1. 上方：RGB 行
    rgb_row = []
    for i in range(NUM_FRAMES):
        rgb_np = (image[i, ..., :3] * 255).astype(np.uint8)
        rgb_row.append(Image.fromarray(rgb_np))

    w, h = rgb_row[0].size
    rgb_canvas = Image.new("RGB", (w * NUM_FRAMES, h))
    for i, im in enumerate(rgb_row):
        rgb_canvas.paste(im, (w * i, 0))

    # 2. 下方：最后一通道 -> 灰度
    ch_row = []
    for i in range(NUM_FRAMES):
        ch_last = (image[i, ..., -1] * 255).astype(np.uint8)
        ch_gray = np.repeat(ch_last[..., None], 3, axis=-1)  # (H,W,3)
        ch_row.append(Image.fromarray(ch_gray))

    ch_canvas = Image.new("RGB", (w * NUM_FRAMES, h))
    for i, im in enumerate(ch_row):
        ch_canvas.paste(im, (w * i, 0))

    # 3. 上下合并
    grid = Image.new("RGB", (w * NUM_FRAMES, h * 3))
    grid.paste(raw_canvas, (0, 0))
    grid.paste(rgb_canvas, (0, h))
    grid.paste(ch_canvas,  (0, 2 * h))
    return grid

# ---------------- 4. Run a few samples ----------------
for idx in range(3):
    dataset._index = idx 
    sample = dataset[idx]
    result_img = infer(sample, seed=42 + idx)
    save_path = os.path.join(COMPARE_DIR, f"result_{idx}.png")
    result_img.save(save_path)
    print(f"Saved: {save_path}")

print("Done!")

/home/viplab/anaconda3/envs/cnet2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading models...


`torch_dtype` is deprecated! Use `dtype` instead!


Warper weights loaded from /data/nd_model_4_wrap_cnet/checkpoint-36000/controlnet/warper.pt
dataset name: /data/train/
folders len 192000
groups len 48000
[debug] idx=0  folders=['/data/train/train_30/2025-08-19_19-35-52', '/data/train/train_30/2025-08-19_19-35-53', '/data/train/train_30/2025-08-19_19-35-54', '/data/train/train_30/2025-08-19_19-35-55']
lkj torch.Size([4, 16])
Running inference: city topview from camera, camera conf as following. {camera_height:100.0,tilt_deg:30.0,lens:100.0}


100%|██████████| 20/20 [00:02<00:00,  7.62it/s]


maxmin tensor(1.3750, device='cuda:0', dtype=torch.float16) tensor(-0.2295, device='cuda:0', dtype=torch.float16)
Saved: ./test/result_0.png
[debug] idx=1  folders=['/data/train/train_30/2025-08-20_20-12-41', '/data/train/train_30/2025-08-20_20-12-42', '/data/train/train_30/2025-08-20_20-12-43', '/data/train/train_30/2025-08-20_20-12-44']
lkj torch.Size([4, 16])
Running inference: city topview from camera, camera conf as following. {camera_height:100.0,tilt_deg:30.0,lens:100.0}


100%|██████████| 20/20 [00:02<00:00,  9.14it/s]


maxmin tensor(1.2012, device='cuda:0', dtype=torch.float16) tensor(-0.1714, device='cuda:0', dtype=torch.float16)
Saved: ./test/result_1.png
[debug] idx=2  folders=['/data/train/train_30/2025-08-20_20-12-45', '/data/train/train_30/2025-08-20_20-12-46', '/data/train/train_30/2025-08-20_20-12-46_2', '/data/train/train_30/2025-08-20_20-12-47']
lkj torch.Size([4, 16])
Running inference: city topview from camera, camera conf as following. {camera_height:100.0,tilt_deg:30.0,lens:100.0}


100%|██████████| 20/20 [00:02<00:00,  9.06it/s]


maxmin tensor(1.2500, device='cuda:0', dtype=torch.float16) tensor(-0.1689, device='cuda:0', dtype=torch.float16)
Saved: ./test/result_2.png
Done!
